# Batched LLM Reranking Experiments (O(N log N) Pairwise Sort)

This notebook runs reranking on the Prime dataset queries. It utilizes the OpenAI Batch API natively. 
Instead of submitting N^2 combinations, we implement an asynchronous layered QuickSort algorithm. 
It requires multiple rounds of polling but reduces API calls to O(N log N) per query.

In [47]:
import os
import re
import ast
import json
import time
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
import sys
sys.path.append('..') # Add root to path to import stark_qa

from stark_qa import load_skb

load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
MODEL = "gpt-4.1-nano-2025-04-14"

# Load KB once for generating document strings
kb = load_skb('prime', download_processed=True)


Loading from /home/coep/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/prime/processed!


## 1. Load the Data Dump & Precalculate Ranks

In [48]:
DATA_DUMP_PATH = "../experiments/prime/LLM_SAVED_RESPONSES/full_data_dump.csv"
df = pd.read_csv(DATA_DUMP_PATH)

def extract_ground_truths(gt_val):
    if pd.isna(gt_val): return []
    if isinstance(gt_val, str):
        try: return ast.literal_eval(gt_val)
        except: return []
    return gt_val

def extract_answer_list(results_str):
    if pd.isna(results_str): return []
    if isinstance(results_str, str):
        try:
            res_dict = ast.literal_eval(results_str)
            return res_dict.get('answer_list', [])[:20]
        except: return []
    return []

df['ground_truths_list'] = df['ground_truths'].apply(extract_ground_truths)
df['top_20_nodes'] = df['results'].apply(extract_answer_list)
print(f"Loaded {len(df)} queries from {DATA_DUMP_PATH}")


Loaded 280 queries from ../experiments/prime/LLM_SAVED_RESPONSES/full_data_dump.csv


## 2. Query Selector

In [49]:
def get_target_queries(df, target_type="best_worst", num_bad=10, num_good=10, query_ids=None):
    def rank_of_first_gt(row):
        gt_set = set(row['ground_truths_list'])
        for i, pred in enumerate(row['top_20_nodes']):
            if pred in gt_set: return i + 1
        return 999
    
    df = df.copy()
    df['first_gt_rank'] = df.apply(rank_of_first_gt, axis=1)
    
    if target_type == "all": return df.to_dict('records')
    if target_type == "specific" and query_ids:
        return df[df['id'].isin(query_ids)].to_dict('records')

    bad_df = df[(df['first_gt_rank'] >= 5) & (df['first_gt_rank'] <= 20)].sort_values(by='first_gt_rank', ascending=False)
    good_df = df[df['first_gt_rank'] == 1]
    
    bad_q = bad_df.head(num_bad).to_dict('records')
    good_q = good_df.head(num_good).to_dict('records')
    
    print(f"Selected {len(bad_q)} bad queries, {len(good_q)} good queries.")
    return bad_q + good_q


## 3. Sort State Machine
Using QuickSort inherently modeled for O(N log N) async execution. Each round submits only comparing Pivot against other members for active unsorted subset chunks.

In [50]:
class QuerySortState:
    def __init__(self, qid, query, docs):
        self.qid = qid
        self.query = query
        self.chunks = [{"type": "UNSORTED", "data": docs}]  # Start with everything unsorted
        
    def get_pending_comparisons(self):
        comps = []
        for chunk in self.chunks:
            if chunk["type"] == "UNSORTED":
                arr = chunk["data"]
                if len(arr) <= 1: continue
                pivot = arr[0]
                for x in arr[1:]:
                    comps.append((self.qid, self.query, pivot, x))
        return comps

    def apply_results(self, results_dict):
        new_chunks = []
        for chunk in self.chunks:
            if chunk["type"] == "SORTED":
                new_chunks.append(chunk)
            else:
                arr = chunk["data"]
                if len(arr) <= 1:
                    new_chunks.append({"type": "SORTED", "data": arr})
                    continue
                
                pivot = arr[0]
                left = []  # Better than pivot (winners go here to rank higher)
                right = [] # Worse or tie
                
                for x in arr[1:]:
                    winner = results_dict.get((self.qid, pivot, x))
                    # If x is chosen as the better doc, x wins and goes to the top (left)
                    if winner == x:
                        left.append(x)
                    else:
                        right.append(x)
                        
                if left: new_chunks.append({"type": "UNSORTED", "data": left})
                new_chunks.append({"type": "SORTED", "data": [pivot]})
                if right: new_chunks.append({"type": "UNSORTED", "data": right})
                    
        self.chunks = new_chunks
        
    def is_done(self):
        return all(c["type"] == "SORTED" for c in self.chunks)
        
    def get_sorted_list(self):
        res = []
        for c in self.chunks:
            res.extend(c["data"])
        return res


## 4. Batch API Executor
Runs rounds of Batch API polling iteratively.

In [51]:
import tiktoken

def submit_and_poll_batch(client, jsonl_path, batch_txt, interval=15):
    print(f"\n-- Uploading {jsonl_path.name}...")
    with jsonl_path.open("rb") as f:
        upload = client.files.create(file=f, purpose="batch")
    
    while True:
        try:
            b = client.batches.create(input_file_id=upload.id, endpoint="/v1/chat/completions", completion_window="24h")
            batch_txt.write_text(b.id)
            print(f"-- Batch created: {b.id}. Polling...")
        except Exception as e:
            if "token_limit" in str(e).lower():
                print(f"   [RATE LIMIT] Enqueue limit hit on create. Sleeping 60s...")
                time.sleep(60)
                continue
            raise e
            
        terms = {"completed", "failed", "expired", "cancelled"}
        while True:
            curr = client.batches.retrieve(b.id)
            c = curr.request_counts
            done, tot, fail = (c.completed, c.total, c.failed) if c else (0,0,0)
            print(f"   [{time.strftime('%H:%M:%S')}] {curr.status} | {done}/{tot} (Failed {fail})", end='\r')
            if curr.status in terms: 
                print() 
                break
            time.sleep(interval)
            
        if curr.status == 'failed':
            err_msg = str(curr.errors).lower()
            if 'token_limit_exceeded' in err_msg or 'limit' in err_msg:
                print(f"   [RATE LIMIT] Batch failed due to enqueued token limit. Sleeping 60s and retrying...")
                time.sleep(60)
                continue # Retry creating the batch
            else:
                raise RuntimeError(f"Batch Failed. Errors: {curr.errors}")
        elif curr.status != 'completed':
            raise RuntimeError(f"Batch ended with status: {curr.status}")
            
        return curr.output_file_id

def map_winner_from_response(ans, node1_id, node2_id):
    try: ans = ans.replace("'", "").replace('"', "").strip() 
    except: pass
    
    if "A" in ans or str(node1_id) in ans: return node1_id
    if "B" in ans or str(node2_id) in ans: return node2_id
    return node1_id # default to pivot tie if invalid

def run_tournament_sort_experiment(queries, experiment_name="batch_reranking_sort"):
    base = Path(f"../experiments/{experiment_name}")
    base.mkdir(parents=True, exist_ok=True)
    
    print("Loading token encoder for accurate batch estimation...")
    try: enc = tiktoken.encoding_for_model(MODEL)
    except: enc = tiktoken.get_encoding("o200k_base")
    
    states = [QuerySortState(q['id'], q['query'], q['top_20_nodes']) for q in queries]
    round_id = 0
    
    while True:
        round_id += 1
        comps = []
        for s in states:
            comps.extend(s.get_pending_comparisons())
            
        if not comps:
            print("All queries fully sorted!")
            break
            
        # Target 2M limit. Use 1.8M to be very safe and ensure we never exceed queue sizes.
        MAX_BATCH_TOKENS = 1_800_000
        comp_chunks = []
        current_chunk = []
        current_chunk_tokens = 0
        
        print(f"\n>>> ROUND {round_id} - Dynamically packing {len(comps)} comparisons within {MAX_BATCH_TOKENS:,} tokens/batch...")
        
        # Prepare and pack dynamically based on literal token sizes
        for qid, query, doc1, doc2 in comps:
            d1_info = kb.get_doc_info(doc1, add_rel=True, compact=True)
            d2_info = kb.get_doc_info(doc2, add_rel=True, compact=True)
            n1_type = kb.get_node_type_by_id(doc1)
            n2_type = kb.get_node_type_by_id(doc2)
            
            prompt = (
                f"The following two elements consist of an ID number, a type and a corresponding descriptive text:\n \n"
                f"{doc1}, {n1_type}, {d1_info}. \n"
                f"{doc2}, {n2_type}, {d2_info}. \n\n"
                f"Find out which of the elements satisfies the following query better: \n"
                f"{query} \n"
                f"Return ONLY the corresponding ID number which corresponds to the element that satisfies "
                f"the given query best. Nothing else."
            )
            req = {
                "custom_id": f"q_{qid}__p_{doc1}__x_{doc2}",
                "method": "POST", "url": "/v1/chat/completions",
                "body": {"model": MODEL, "messages": [{"role": "user", "content": prompt}], "temperature": 0.0, "max_tokens": 10}
            }
            
            req_str = json.dumps(req)
            req_tokens = len(enc.encode(req_str))
            
            if current_chunk_tokens + req_tokens > MAX_BATCH_TOKENS and current_chunk:
                comp_chunks.append(current_chunk)
                current_chunk = []
                current_chunk_tokens = 0
                
            current_chunk.append(req_str)
            current_chunk_tokens += req_tokens
            
        if current_chunk:
            comp_chunks.append(current_chunk)
            
        results_dict = {}
        for part_idx, comp_chunk in enumerate(comp_chunks):
            suffix = f"_part_{part_idx+1}" if len(comp_chunks) > 1 else ""
            req_path = base / f"round_{round_id}{suffix}_req.jsonl"
            res_path = base / f"round_{round_id}{suffix}_res.jsonl"
            batch_txt = base / f"round_{round_id}{suffix}.txt"
            
            if len(comp_chunks) > 1:
                print(f"\n  --- Submitting part {part_idx+1}/{len(comp_chunks)} ({len(comp_chunk)} queries) ---")
            
            with open(req_path, 'w', encoding='utf-8') as f:
                for req_str in comp_chunk:
                    f.write(req_str + "\n")
                    
            out_id = submit_and_poll_batch(client, req_path, batch_txt)
            client.files.content(out_id).write_to_file(res_path)
            
            with open(res_path, 'r') as f:
                for line in f:
                    data = json.loads(line)
                    cid = data["custom_id"]
                    parts = cid.replace("q_", "").split("__")
                    qid = int(parts[0])
                    d1 = int(parts[1].replace("p_",""))
                    d2 = int(parts[2].replace("x_",""))
                    
                    try:
                        ans = data["response"]["body"]["choices"][0]["message"]["content"]
                        winner = map_winner_from_response(ans, d1, d2)
                    except:
                        winner = d1
                    results_dict[(qid, d1, d2)] = winner
                    
        # Apply Results to the State
        for s in states:
            s.apply_results(results_dict)
            
    # Wrap up outputs
    final_queries = []
    state_dict = {s.qid: s.get_sorted_list() for s in states}
    for q in queries:
        output = q.copy()
        output['reranked'] = state_dict[q['id']]
        final_queries.append(output)
        
    return final_queries


## 5. View Metrics

In [52]:
import datetime
def analyze_metrics(final_queries, save_dir="../experiments/batch_reranking_results", description="pairwise_tournament_sort"):
    metrics = []
    def mrr(lst, gts):
        for i, v in enumerate(lst): 
            if v in gts: return 1.0/(i+1)
        return 0.0
        
    for q in final_queries:
        gts = set(q['ground_truths_list'])
        orig, rrk = q['top_20_nodes'], q['reranked']
        
        metrics.append({
            'id': q['id'],
            'orig_mrr': mrr(orig, gts), 'new_mrr': mrr(rrk, gts),
            'orig_h1': 1.0 if orig and orig[0] in gts else 0.0,
            'new_h1': 1.0 if rrk and rrk[0] in gts else 0.0,
            'orig_h5': 1.0 if set(orig[:5]) & gts else 0.0,
            'new_h5': 1.0 if set(rrk[:5]) & gts else 0.0
        })
        
    df_m = pd.DataFrame(metrics)
    df_m['mrr_diff'] = df_m['new_mrr'] - df_m['orig_mrr']
    
    out_str = "=== RERANKING RESULTS ===\n"
    out_str += f"Total Queries Run: {len(df_m)}\n"
    out_str += f"Method: {description}\n"
    out_str += f"MRR:      {df_m['orig_mrr'].mean():.3f} -> {df_m['new_mrr'].mean():.3f} (Δ {df_m['mrr_diff'].mean():+.3f})\n"
    out_str += f"Hit@1:    {df_m['orig_h1'].mean():.3f} -> {df_m['new_h1'].mean():.3f}\n"
    out_str += f"Hit@5:    {df_m['orig_h5'].mean():.3f} -> {df_m['new_h5'].mean():.3f}\n"
    print(out_str)
    
    # Save results to folders
    os.makedirs(save_dir, exist_ok=True)
    ts = datetime.datetime.now().strftime('%Y%md_%H%M%S')
    csv_path = os.path.join(save_dir, f"detailed_metrics_{description}_{ts}.csv")
    txt_path = os.path.join(save_dir, f"result_summary_{description}_{ts}.txt")
    
    df_m.to_csv(csv_path, index=False)
    with open(txt_path, 'w') as f:
        f.write(out_str)
    print(f"Saved detailed metrics to: {csv_path}")
    print(f"Saved summary to: {txt_path}")
    return df_m


## 6. Experiment Wrapper

In [53]:
def run_experiment(queries, method="pairwise", experiment_name="experiment_1"):
    print(f"\n{'='*50}")
    print(f"Starting Experiment: {experiment_name} | Method: {method.upper()} | Queries: {len(queries)}")
    print(f"{'='*50}\n")
    
    if method == "pairwise":
        outputs = run_tournament_sort_experiment(queries, experiment_name=experiment_name)
    elif method == "pointwise":
        # Assuming a pointwise function exists or will be added
        if 'run_pointwise_experiment' in globals():
            outputs = run_pointwise_experiment(queries)
        else:
            raise NotImplementedError("run_pointwise_experiment is not defined yet.")
    else:
        raise ValueError(f"Unknown method: {method}")
        
    analysis_df = analyze_metrics(outputs, description=experiment_name)
    return outputs, analysis_df

## 6. Run on All Queries (Batch)

In [54]:
# Get all queries (280 total usually for prime dataset test splits etc)
all_queries = get_target_queries(df, target_type="all")
print(f"Running tournament sort on {len(all_queries)} queries...")

outputs_all = run_tournament_sort_experiment(test_queries_bw, experiment_name="pairwise_tournament_all_280")
analysis_df_all = analyze_metrics(outputs_all, description="pairwise_tournament_all_280")

Running tournament sort on 280 queries...
Loading token encoder for accurate batch estimation...

>>> ROUND 1 - Dynamically packing 740 comparisons within 1,800,000 tokens/batch...


KeyboardInterrupt: 

In [55]:
import tiktoken

def estimate_token_cost(queries, method="pairwise", model="gpt-4o-mini"):
    """
    Estimates the absolute maximum token count and cost for a batch operation.
    """
    try: enc = tiktoken.encoding_for_model(model)
    except: enc = tiktoken.get_encoding("o200k_base") # Default to standard modern encoding
    
    total_prompt_tokens = 0
    
    # For Pairwise we estimate N(N-1)/2 max comparisons * token sizes
    # even though tournament averages O(N log N), batch limits apply to what gets queued.
    if method == "pairwise":
        for q in queries:
            docs = q['top_20_nodes']
            if not docs: continue
            
            # Estimate size of average two docs for this query
            d1 = docs[0]
            d1_info = kb.get_doc_info(d1, add_rel=True, compact=True)
            
            sample_prompt = (
                f"The following two elements consist of an ID number, a type and a corresponding descriptive text:\n \n"
                f"{d1}, x, {d1_info}. \n"
                f"{d1}, x, {d1_info}. \n\n"
                f"Find out which of the elements satisfies the following query better: \n"
                f"{q['query']} \n"
                f"Return ONLY the corresponding ID number which corresponds to the element that satisfies "
                f"the given query best. Nothing else."
            )
            
            # QuickSort average depth is ~4.3 for 20 elements, so roughly ~60 pairs instead of 190. 
            # But we'll estimate upper bound for a single batch round comparing pivot vs everything (N-1 = 19).
            expected_comparisons_first_round = len(docs) - 1
            
            base_tokens = len(enc.encode(sample_prompt))
            total_prompt_tokens += (base_tokens * expected_comparisons_first_round)
            
    print(f"--- Token Estimate for {method.upper()} ({len(queries)} queries) ---")
    print(f"Estimated tokens per batch round: {total_prompt_tokens:,}")
    # Assume mini $0.15 / 1M tokens, standard gpt-4.5 is $75 / 1M
    rate = 0.025 if 'nano' in model else 2.50  
    print(f"Estimated cost per round: ${(total_prompt_tokens / 1_000_000) * rate:.4f}")
    print("Note: The Batch API gives you a 50% discount on these rates.")
    return total_prompt_tokens

test_queries_bw = get_target_queries(df, target_type="best_worst", num_bad=140, num_good=140)
estimate_token_cost(test_queries_bw, method="pairwise", model=MODEL)


Selected 39 bad queries, 102 good queries.


--- Token Estimate for PAIRWISE (141 queries) ---
Estimated tokens per batch round: 19,222,003
Estimated cost per round: $0.4806
Note: The Batch API gives you a 50% discount on these rates.


19222003

## 7. Run on Test Queries


In [56]:
# EXPERIMENT OPTIONS:
# final_queries, df_metrics = run_experiment(all_queries, method="pairwise", experiment_name="pairwise_all")
# final_queries, df_metrics = run_experiment(all_queries, method="pointwise", experiment_name="pointwise_all")
test_queries_bw = get_target_queries(df, target_type="best_worst", num_bad=20, num_good=20)
outputs_test, analysis_df_test = run_experiment(test_queries_bw, method="pairwise", experiment_name="pairwise_test_40_nano")

# Remember `all_queries = get_target_queries(df, target_type="all")` pulls the 280-query chunk!


Selected 20 bad queries, 20 good queries.

Starting Experiment: pairwise_test_40_nano | Method: PAIRWISE | Queries: 40

Loading token encoder for accurate batch estimation...

>>> ROUND 1 - Dynamically packing 740 comparisons within 1,800,000 tokens/batch...

  --- Submitting part 1/4 (137 queries) ---

-- Uploading round_1_part_1_req.jsonl...
-- Batch created: batch_69b676b7e26881908c4929d51e1af085. Polling...
   [14:38:51] completed | 137/137 (Failed 0)))

  --- Submitting part 2/4 (183 queries) ---

-- Uploading round_1_part_2_req.jsonl...


BadRequestError: Error code: 400 - {'error': {'message': 'Billing hard limit has been reached', 'type': 'invalid_request_error', 'param': None, 'code': 'billing_hard_limit_reached'}}